In [1]:
from datasets import load_dataset

ds = load_dataset("eagle0504/openai-gsm8k-enhanced-using-together-ai-deepseek-train8k-test1k-v1")

/home/u12321044/anaconda3/envs/py39/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating test split: 100%|██████████| 1319/1319 [00:00<00:00, 118489.76 examples/s]


In [7]:
from datasets import load_dataset
import numpy as np

# 加载数据集
dataset = load_dataset("eagle0504/openai-gsm8k-enhanced-using-together-ai-deepseek-train8k-test1k-v1", 'default')

# 获取问题和解答字段
questions = dataset['train']['question']
answers = dataset['train']['cot']

# 计算样本数量
sample_count = len(questions)

# 计算问题和答案的长度（这里以单词计数为例）
question_lengths = [len(q.split()) for q in questions]
answer_lengths = [len(a.split()) for a in answers]

# 基础统计信息
avg_question_length = np.mean(question_lengths)
min_question_length = min(question_lengths)
max_question_length = max(question_lengths)

avg_answer_length = np.mean(answer_lengths)
min_answer_length = min(answer_lengths)
max_answer_length = max(answer_lengths)

# 计算独特词汇量
vocab = set()
for text in questions + answers:
    vocab.update(text.split())
vocab_size = len(vocab)

# 打印结果
print(f"样本数量: {sample_count}")
print(f"问题长度 - 平均: {avg_question_length}, 最小: {min_question_length}, 最大: {max_question_length}")
print(f"解答长度 - 平均: {avg_answer_length}, 最小: {min_answer_length}, 最大: {max_answer_length}")
print(f"独特词汇量: {vocab_size}")

样本数量: 7473
问题长度 - 平均: 45.092600026763016, 最小: 9, 最大: 183
解答长度 - 平均: 171.9177037334404, 最小: 1, 最大: 517
独特词汇量: 64058


In [ ]:

# 查看训练集的前5条数据
for i in range(2):
    print(f"样本 {i+1}:")
    print(f"COT: {dataset['train'][i]['cot']}")
    print(f"问题: {dataset['train'][i]['question']}")
    print(f"解答: {dataset['train'][i]['answer']}")
    print("-"*50)  # 分隔线，便于区分不同样本

样本 1:
数据: <think>To solve the problem, we need to determine the total number of clips Natalia sold in April and May. Here's the reasoning:

1. **Clips sold in April**: Natalia sold 48 clips in April. This is given directly in the problem.

2. **Clips sold in May**: Natalia sold half as many clips in May as she did in April. To find this, we divide the number of clips sold in April by 2:
   \[
   \text{Clips in May} = \frac{48}{2} = 24
   \]

3. **Total clips sold**: To find the total number of clips sold in April and May, we add the number of clips sold in each month:
   \[
   \text{Total clips} = 48 + 24 = 72
   \]

Therefore, Natalia sold a total of 72 clips in April and May.</think>
<response>#### 72</response>
问题: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
解答: Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April an

In [8]:
# 转为sft格式
import json

# 原始数据文件路径和目标文件路径
input_file = "/home/u12321044/share/liang_52/align_tax/gsm8k_cot_aligned_answers_vllm.jsonl"  # 原始数据文件
output_file = "/home/u12321044/share/liang_52/align_tax/LLaMA-Factory/data/gsm8k_cot.json"  # 转换后的数据文件

# 读取原始数据
with open(input_file, "r", encoding="utf-8") as f:
    raw_data = [json.loads(line) for line in f]

# 转换数据格式
formatted_data = []
for sample in raw_data:
    formatted_sample = {
        "instruction": sample["question"],  # instruction 对应 question
        "input": "",  # input 始终为空字符串
        "output": sample["original_cot"]  # output 对应 original_answer
    }
    formatted_data.append(formatted_sample)

# 保存转换后的数据到新的 JSON 文件
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(formatted_data, f, ensure_ascii=False, indent=2)

print(f"数据已成功转换并保存到 {output_file}")

数据已成功转换并保存到 /home/u12321044/share/liang_52/align_tax/LLaMA-Factory/data/gsm8k_cot.json


In [9]:
# 生成test数据
from datasets import load_dataset
import json

# 加载GSM8K数据集
dataset = load_dataset("eagle0504/openai-gsm8k-enhanced-using-together-ai-deepseek-train8k-test1k-v1", 'default')

# 提取测试集
test_data = dataset['test']

# 转换数据格式
formatted_data = []
for sample in test_data:
    formatted_sample = {
        "instruction": sample["question"],  # instruction 对应 question
        "input": "",  # input 始终为空字符串
        "output": sample["cot"]  # output 对应 answer, 注意这里直接使用了"answer"字段，如果你的数据中有"original_answer"请相应替换
    }
    formatted_data.append(formatted_sample)

# 目标文件路径
output_file = "/home/u12321044/share/liang_52/align_tax/LLaMA-Factory/data/gsm8k_cot_test.json"

# 保存转换后的数据到新的 JSON 文件
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(formatted_data, f, ensure_ascii=False, indent=2)

print(f"数据已成功转换并保存到 {output_file}")

数据已成功转换并保存到 /home/u12321044/share/liang_52/align_tax/LLaMA-Factory/data/gsm8k_cot_test.json


In [10]:
# 转为self-style-sft格式
import json

# 原始数据文件路径和目标文件路径
input_file = "/home/u12321044/share/liang_52/align_tax/gsm8k_cot_aligned_answers_vllm.jsonl"  # 原始数据文件
output_file = "/home/u12321044/share/liang_52/align_tax/LLaMA-Factory/data/gsm8k_align_cot.json"  # 转换后的数据文件

# 读取原始数据
with open(input_file, "r", encoding="utf-8") as f:
    raw_data = [json.loads(line) for line in f]

# 转换数据格式
formatted_data = []
for sample in raw_data:
    formatted_sample = {
        "instruction": sample["question"],  # instruction 对应 question
        "input": "",  # input 始终为空字符串
        "output": sample["aligned_cot"]  # output 对应 original_answer
    }
    formatted_data.append(formatted_sample)

# 保存转换后的数据到新的 JSON 文件
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(formatted_data, f, ensure_ascii=False, indent=2)

print(f"数据已成功转换并保存到 {output_file}")

数据已成功转换并保存到 /home/u12321044/share/liang_52/align_tax/LLaMA-Factory/data/gsm8k_align_cot.json


## TIR数据集处理

In [1]:
from datasets import load_dataset

ds = load_dataset("AI-MO/NuminaMath-TIR")

/home/u12321044/anaconda3/envs/py39/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating test split: 100%|██████████| 99/99 [00:00<00:00, 10619.03 examples/s]


In [3]:
from datasets import load_dataset
import numpy as np

# 加载数据集
dataset = load_dataset("AI-MO/NuminaMath-TIR", 'default')

# 获取问题和解答字段
questions = dataset['train']['problem']
answers = dataset['train']['solution']

# 计算样本数量
sample_count = len(questions)

# 计算问题和答案的长度（这里以单词计数为例）
question_lengths = [len(q.split()) for q in questions]
answer_lengths = [len(a.split()) for a in answers]

# 基础统计信息
avg_question_length = np.mean(question_lengths)
min_question_length = min(question_lengths)
max_question_length = max(question_lengths)

avg_answer_length = np.mean(answer_lengths)
min_answer_length = min(answer_lengths)
max_answer_length = max(answer_lengths)

# 计算独特词汇量
vocab = set()
for text in questions + answers:
    vocab.update(text.split())
vocab_size = len(vocab)

# 打印结果
print(f"样本数量: {sample_count}")
print(f"问题长度 - 平均: {avg_question_length}, 最小: {min_question_length}, 最大: {max_question_length}")
print(f"解答长度 - 平均: {avg_answer_length}, 最小: {min_answer_length}, 最大: {max_answer_length}")
print(f"独特词汇量: {vocab_size}")

样本数量: 72441
问题长度 - 平均: 37.56142239891774, 最小: 1, 最大: 429
解答长度 - 平均: 347.70175729214117, 最小: 23, 最大: 1896
独特词汇量: 879700


In [5]:

# 查看训练集的前5条数据
for i in range(2):
    print(f"样本 {i+1}:")
    print(f"COT: {dataset['train'][i]['problem']}")
    print(f"问题: {dataset['train'][i]['solution']}")
    # print(f"解答: {dataset['train'][i]['answer']}")
    print("-"*50)  # 分隔线，便于区分不同样本

样本 1:
COT: What is the coefficient of $x^2y^6$ in the expansion of $\left(\frac{3}{5}x-\frac{y}{2}\right)^8$?  Express your answer as a common fraction.
问题: To determine the coefficient of \(x^2y^6\) in the expansion of \(\left(\frac{3}{5}x - \frac{y}{2}\right)^8\), we can use the binomial theorem.

The binomial theorem states:
\[
(a + b)^n = \sum_{k=0}^{n} \binom{n}{k} a^{n-k} b^k
\]

In this case, \(a = \frac{3}{5}x\), \(b = -\frac{y}{2}\), and \(n = 8\).

We are interested in the term that contains \(x^2y^6\). In the general term of the binomial expansion:
\[
\binom{8}{k} \left(\frac{3}{5}x\right)^{8-k} \left(-\frac{y}{2}\right)^k
\]

To get \(x^2\), we need \(8 - k = 2\), thus \(k = 6\).

Substituting \(k = 6\) into the expression:
\[
\binom{8}{6} \left(\frac{3}{5}x\right)^{8-6} \left(-\frac{y}{2}\right)^6 = \binom{8}{6} \left(\frac{3}{5}x\right)^2 \left(-\frac{y}{2}\right)^6
\]

Now, we will compute each part of this expression.

1. Calculate the binomial coefficient \(\binom{8}{6

In [1]:
# 转为sft格式
import json

# 原始数据文件路径和目标文件路径
input_file = "/home/u12321044/share/liang_52/align_tax/Tir_cot_aligned_answers_vllm.jsonl"  # 原始数据文件
output_file = "/home/u12321044/share/liang_52/align_tax/LLaMA-Factory/data/Tir_cot.json"  # 转换后的数据文件

# 读取原始数据
with open(input_file, "r", encoding="utf-8") as f:
    raw_data = [json.loads(line) for line in f]

# 转换数据格式
formatted_data = []
for sample in raw_data:
    formatted_sample = {
        "instruction": sample["question"],  # instruction 对应 question
        "input": "",  # input 始终为空字符串
        "output": sample["original_cot"]  # output 对应 original_answer
    }
    formatted_data.append(formatted_sample)

# 保存转换后的数据到新的 JSON 文件
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(formatted_data, f, ensure_ascii=False, indent=2)

print(f"数据已成功转换并保存到 {output_file}")

数据已成功转换并保存到 /home/u12321044/share/liang_52/align_tax/LLaMA-Factory/data/Tir_cot.json


In [2]:
# 转为self-style-sft格式
import json

# 原始数据文件路径和目标文件路径
input_file = "/home/u12321044/share/liang_52/align_tax/Tir_cot_aligned_answers_vllm.jsonl"  # 原始数据文件
output_file = "/home/u12321044/share/liang_52/align_tax/LLaMA-Factory/data/Tir_align_cot.json"  # 转换后的数据文件

# 读取原始数据
with open(input_file, "r", encoding="utf-8") as f:
    raw_data = [json.loads(line) for line in f]

# 转换数据格式
formatted_data = []
for sample in raw_data:
    formatted_sample = {
        "instruction": sample["question"],  # instruction 对应 question
        "input": "",  # input 始终为空字符串
        "output": sample["aligned_cot"]  # output 对应 original_answer
    }
    formatted_data.append(formatted_sample)

# 保存转换后的数据到新的 JSON 文件
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(formatted_data, f, ensure_ascii=False, indent=2)

print(f"数据已成功转换并保存到 {output_file}")

数据已成功转换并保存到 /home/u12321044/share/liang_52/align_tax/LLaMA-Factory/data/Tir_align_cot.json


In [1]:
# 生成Tir数据的test

from datasets import load_dataset
import json

# 加载GSM8K数据集
dataset = load_dataset("AI-MO/NuminaMath-TIR", 'default')

# 提取测试集
test_data = dataset['test']

# 转换数据格式
formatted_data = []
for sample in test_data:
    formatted_sample = {
        "instruction": sample["problem"],  # instruction 对应 question
        "input": "",  # input 始终为空字符串
        "output": sample["solution"]  # output 对应 answer, 注意这里直接使用了"answer"字段，如果你的数据中有"original_answer"请相应替换
    }
    formatted_data.append(formatted_sample)

# 目标文件路径
output_file = "/home/u12321044/share/liang_52/align_tax/LLaMA-Factory/data/Tir_cot_test.json"

# 保存转换后的数据到新的 JSON 文件
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(formatted_data, f, ensure_ascii=False, indent=2)

print(f"数据已成功转换并保存到 {output_file}")

/home/u12321044/anaconda3/envs/py39/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


数据已成功转换并保存到 /home/u12321044/share/liang_52/align_tax/LLaMA-Factory/data/Tir_cot_test.json


In [2]:
# openr1-math 数据集处理
from datasets import load_dataset
import numpy as np

# 加载数据集
dataset = load_dataset("open-r1/OpenR1-Math-220k", "default")

# 获取问题和解答字段
questions = dataset['train']['problem']
answers = dataset['train']['solution']

# 计算样本数量
sample_count = len(questions)

# 计算问题和答案的长度（这里以单词计数为例）
question_lengths = [len(q.split()) for q in questions]
answer_lengths = [len(a.split()) for a in answers]

# 基础统计信息
avg_question_length = np.mean(question_lengths)
min_question_length = min(question_lengths)
max_question_length = max(question_lengths)

avg_answer_length = np.mean(answer_lengths)
min_answer_length = min(answer_lengths)
max_answer_length = max(answer_lengths)

# 计算独特词汇量
vocab = set()
for text in questions + answers:
    vocab.update(text.split())
vocab_size = len(vocab)

# 打印结果
print(f"样本数量: {sample_count}")
print(f"问题长度 - 平均: {avg_question_length}, 最小: {min_question_length}, 最大: {max_question_length}")
print(f"解答长度 - 平均: {avg_answer_length}, 最小: {min_answer_length}, 最大: {max_answer_length}")
print(f"独特词汇量: {vocab_size}")

样本数量: 93733
问题长度 - 平均: 46.11890156081636, 最小: 2, 最大: 2046
解答长度 - 平均: 149.23455986685585, 最小: 0, 最大: 2929
独特词汇量: 1122708


In [8]:
dataset = load_dataset("open-r1/OpenR1-Math-220k", "default")
split_cnt = 200
# 提取测试集
test_problems = dataset['train']['problem'][(sample_count-split_cnt):]
test_solutions = dataset['train']['solution'][(sample_count-split_cnt):]

# 转换数据格式
formatted_data = []
for i in range(len(test_problems)):
    formatted_sample = {
        "instruction": test_problems[i],  # instruction 对应 question
        "input": "",  # input 始终为空字符串
        "output": test_solutions[i]  # output 对应 answer, 注意这里直接使用了"answer"字段，如果你的数据中有"original_answer"请相应替换
    }
    formatted_data.append(formatted_sample)

# 目标文件路径
output_file = "/home/u12321044/share/liang_52/align_tax/LLaMA-Factory/data/openr1_cot_test.json"

# 保存转换后的数据到新的 JSON 文件
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(formatted_data, f, ensure_ascii=False, indent=2)

print(f"数据已成功转换并保存到 {output_file}")

# 提取测试集
train_problems = dataset['train']['problem'][:(sample_count-split_cnt)]
train_solutions = dataset['train']['solution'][:(sample_count-split_cnt)]
# 转换数据格式
formatted_data = []
for i in range(len(train_problems)):
    formatted_sample = {
        "instruction": train_problems[i],  # instruction 对应 question
        "input": "",  # input 始终为空字符串
        "output": train_solutions[i]  # output 对应 answer, 注意这里直接使用了"answer"字段，如果你的数据中有"original_answer"请相应替换
    }
    formatted_data.append(formatted_sample)

# 目标文件路径
output_file = "/home/u12321044/share/liang_52/align_tax/LLaMA-Factory/data/openr1_cot.json"

# 保存转换后的数据到新的 JSON 文件
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(formatted_data, f, ensure_ascii=False, indent=2)

print(f"数据已成功转换并保存到 {output_file}")

数据已成功转换并保存到 /home/u12321044/share/liang_52/align_tax/LLaMA-Factory/data/openr1_cot_test.json
数据已成功转换并保存到 /home/u12321044/share/liang_52/align_tax/LLaMA-Factory/data/openr1_cot.json


In [ ]:
# 将policy model的前k个token拼到训练数据上
# 首先VLLM生成policy model answer，LLaMA-Factory运行bash run_test_vllm.sh，然后将生成的answer拼到原始数据上

